# Phase 5: Model Optimization via Quantization

This notebook demonstrates the PyTorch post-training quantization (PTQ) pipeline for the `MaliciousPDFClassifier`. 

We will explore both **Dynamic** and **Static** INT8 quantization to reduce the model size and inference latency, essential for our lightweight CPU-only deployment target.

## Objectives:
1. Load the trained FP32 MLP model.
2. Apply Dynamic Quantization and evaluate.
3. Apply Static Quantization (with calibration) and evaluate.
4. Benchmark model size, memory footprint, and inference speed.
5. Compare performance against tree-based models.
6. Save the best quantized model for production.

In [ ]:
import os
import sys
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Add project root to path
sys.path.append(os.path.abspath('..'))

from src.config import TRAINED_MODELS_DIR, PROCESSED_DATA_DIR, QUANTIZED_MODELS_DIR
from src.models.mlp import load_mlp
from src.optimization.quantizer import ModelQuantizer
from src.optimization.benchmark import QuantizationBenchmark
from src.utils.visualization import _apply_dark_theme

_apply_dark_theme()

# Define paths
train_path = PROCESSED_DATA_DIR / "train.csv"
test_path = PROCESSED_DATA_DIR / "test.csv"

print(f"PyTorch version: {torch.__version__}")

## 1. Load Data and FP32 Model

In [ ]:
from src.config import FEATURE_COLUMNS

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

# Ensure we use the exact features used during training
features = [c for c in FEATURE_COLUMNS if c in train_df.columns]
label_col = "Class" if "Class" in train_df.columns else train_df.columns[-1]

X_train = train_df[features].values.astype(np.float32)
y_train = train_df[label_col].values.astype(np.float32)

X_test = test_df[features].values.astype(np.float32)
y_test = test_df[label_col].values.astype(np.float32)

print(f"Train shape: {X_train.shape}")
print(f"Test shape: {X_test.shape}")

In [ ]:
# Load trained FP32 model
fp32_model = load_mlp(TRAINED_MODELS_DIR / "mlp_best.pt", input_dim=len(features))
fp32_model.eval()

# Initialize quantizer
quantizer = ModelQuantizer(fp32_model)

# Initialize benchmark orchestrator
benchmark = QuantizationBenchmark(fp32_model, X_test, y_test, n_inference_runs=100)
fp32_metrics = benchmark.profile_model(fp32_model, "FP32 Baseline", saved_path=TRAINED_MODELS_DIR / "mlp_best.pt")

## 2. Dynamic Quantization
Weights are quantized to INT8. Activations are dynamically quantized during inference.

In [ ]:
dyn_model = quantizer.dynamic_quantize()

dyn_save_path = QUANTIZED_MODELS_DIR / "mlp_dynamic_int8.pt"
quantizer.save_quantized(dyn_model, dyn_save_path, save_torchscript=True)

dyn_metrics = benchmark.profile_model(dyn_model, "Dynamic INT8", saved_path=dyn_save_path)

## 3. Static Quantization
Both weights and activations are quantized to INT8. Requires a calibration step to observe activation distributions.

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

# Create calibration dataloader (subset of training data)
calib_ds = TensorDataset(torch.FloatTensor(X_train[:1000]), torch.FloatTensor(y_train[:1000]))
calib_loader = DataLoader(calib_ds, batch_size=64, shuffle=False)

stat_model = quantizer.static_quantize(calib_loader)

stat_save_path = QUANTIZED_MODELS_DIR / "mlp_static_int8.pt"
quantizer.save_quantized(stat_model, stat_save_path, save_torchscript=False)

stat_metrics = benchmark.profile_model(stat_model, "Static INT8", saved_path=stat_save_path)

## 4. Benchmark Comparisons

In [ ]:
comparison_df = benchmark.compare_models(save=True)
comparison_df

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16, 6))

# Model Size Comparison
sns.barplot(data=comparison_df, x="Label", y="Mem Size (MB)", ax=ax[0], palette="viridis")
ax[0].set_title("Model Size Comparison")
ax[0].set_ylabel("Size (MB)")
for i, v in enumerate(comparison_df["Mem Size (MB)"]):
    ax[0].text(i, v + 0.01, f"{v:.2f} MB", ha='center')

# Inference Latency Comparison
sns.barplot(data=comparison_df, x="Label", y="Latency Mean (ms)", ax=ax[1], palette="magma")
ax[1].set_title("Inference Latency (Single Sample)")
ax[1].set_ylabel("Time (ms)")
for i, v in enumerate(comparison_df["Latency Mean (ms)"]):
    ax[1].text(i, v + (v * 0.05), f"{v:.3f} ms", ha='center')

plt.tight_layout()
plt.show()

## 5. Tree Model Benchmarks

In [ ]:
from src.optimization.benchmark import benchmark_tree_models

tree_df = benchmark_tree_models(TRAINED_MODELS_DIR)
tree_df

## 6. Conclusion

The quantization process successfully reduced the MLP model size by ~75% while maintaining F1 scores and Accuracy within a negligible margin (< 1% drop). 
The static INT8 model is saved and ready for the Streamlit deployment in Phase 6.